In [1]:
# ==========================================================
# ADIM 1: Kurulum ve Kütüphaneler
# ==========================================================
# Hocanın koduyla aynı
!pip install transformers datasets -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

print("Adım 1: Gerekli kütüphaneler (transformers, torch, pandas) yüklendi.")


# ==========================================================
# ADIM 2: Veri Yükleme ve Hazırlama (SENİN VERİ SETİN)
# ==========================================================
# Bu bölüm, hocanın 'load_dataset("imdb")' kısmını
# senin 'clean_data222.csv' dosyanla değiştirir.

yerel_dosya_yolu = "/content/drive/MyDrive/Colab Notebooks/clean_data222.csv"
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'label'

try:
    print(f"\nAdım 2: '{yerel_dosya_yolu}' dosyasından veri seti yükleniyor...")
    df = pd.read_csv(yerel_dosya_yolu)

    # 'clean_content' sütununu 'text' olarak yeniden adlandıralım
    if 'clean_content' in df.columns:
        df.rename(columns={'clean_content': 'text'}, inplace=True)

    print(f"Veri boyutu (filtresiz): {df.shape}")

    # 1. Etiketleri Metinden Tamsayıya Dönüştürme (6 Sınıf için)
    # BERT modeli de CNN gibi 'Politics' yerine 0, 1, 2... bekler
    labels_list = sorted(df[LABEL_COLUMN].unique())
    label_to_int = {label: i for i, label in enumerate(labels_list)}
    # Tahmin fonksiyonunda kullanmak için tersini de oluşturalım
    int_to_label = {i: label for label, i in label_to_int.items()}

    df['label_int'] = df[LABEL_COLUMN].map(label_to_int)

    # Bu değişkenler sonraki adımlarda kullanılacak
    num_classes = len(label_to_int) # Toplam sınıf sayısı (6)

    print(f"\n{num_classes} sınıf bulundu ve sayısallaştırıldı:")
    print(label_to_int)
    print("-" * 30)

    # 2. Veriyi Eğitim ve Test Setlerine Ayırma
    X = df[TEXT_COLUMN].fillna('').values # NaN hatalarını önle
    y = df['label_int'].values

    # stratify=y: 6 sınıfın oranını korur (CNN'deki gibi)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    print(f"Eğitim Seti (X_train) Boyutu: {len(X_train)}")
    print(f"Test Seti (X_test) Boyutu: {len(X_test)}")
    print(f"Veri hazırlama (Adım 2) tamamlandı.")
    print("\n--- Adım 1 ve 2 tamamlandı. Şimdi Adım 3'e (Dataset Sınıfı) geçebilirsiniz. ---")


except Exception as e:
    print(f"\n\nHATA: Veri işlenirken bir sorun oluştu: {e}")

Adım 1: Gerekli kütüphaneler (transformers, torch, pandas) yüklendi.

Adım 2: '/content/drive/MyDrive/Colab Notebooks/clean_data222.csv' dosyasından veri seti yükleniyor...
Veri boyutu (filtresiz): (136124, 2)

6 sınıf bulundu ve sayısallaştırıldı:
{'Emotion': 0, 'Financial': 1, 'Health': 2, 'Politics': 3, 'Science': 4, 'Sport': 5}
------------------------------
Eğitim Seti (X_train) Boyutu: 108899
Test Seti (X_test) Boyutu: 27225
Veri hazırlama (Adım 2) tamamlandı.

--- Adım 1 ve 2 tamamlandı. Şimdi Adım 3'e (Dataset Sınıfı) geçebilirsiniz. ---


In [4]:
# ==========================================================
# ADIM 3: BERT Tokenizer ve Dataset Sınıfı
# ==========================================================
# Adım 1+2 bloğundaki 'X_train', 'y_train' vb. değişkenleri kullanır.

try:
    # 1. BERT Tokenizer'ı Yükleme
    # Bu, 'bert-base-uncased' modelinin sözlüğünü (vocabulary)
    # ve metni parçalama (tokenize) kurallarını indirir.
    print("Adım 3: BERT Tokenizer ('bert-base-uncased') yükleniyor...")
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    print("Tokenizer başarıyla yüklendi.")


    # 2. Dataset Sınıfını Tanımlama
    class NewsBertDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=256):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            # max_len=512 (hocanın kodu) 'cpu' için çok yavaş olabilir.
            # 256 daha makul bir başlangıçtır.
            self.max_len = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            text = str(self.texts[idx])
            label = self.labels[idx]

            # Hocanın kodundaki gibi tokenizer'ı kullanarak
            # metni BERT formatına (input_ids, attention_mask) dönüştür
            encoding = self.tokenizer(
                text,
                add_special_tokens=True, # [CLS] ve [SEP] ekle
                max_length=self.max_len, # max_len'e göre doldur/kırp
                padding='max_length',    # 'max_length'e kadar doldur
                truncation=True,         # 'max_length'ten uzunsa kırp
                return_tensors='pt'      # PyTorch tensörü olarak döndür
            )

            # Hocanın koduyla aynı formatta bir sözlük (dictionary) döndür
            return {
                'input_ids': encoding['input_ids'].flatten(),
                'attention_mask': encoding['attention_mask'].flatten(),
                'labels': torch.tensor(label, dtype=torch.long)
            }

    # --- Test ---
    print("\nNewsBertDataset sınıfı oluşturuluyor...")

    # max_len'i 256 olarak ayarlayalım (CPU'da daha hızlı çalışması için)
    MAX_LEN = 256

    train_dataset_bert = NewsBertDataset(X_train, y_train, tokenizer, max_len=MAX_LEN)
    test_dataset_bert = NewsBertDataset(X_test, y_test, tokenizer, max_len=MAX_LEN)

    print(f"Dataset'ler MAX_LEN={MAX_LEN} ile oluşturuldu.")

    # Bir adet örnek alıp bakalım:
    sample = train_dataset_bert[0]
    print(f"\nEğitim setinden ilk örnek (BERT formatında):")
    print(f"  input_ids shape: {sample['input_ids'].shape}") # [256] olmalı
    print(f"  attention_mask shape: {sample['attention_mask'].shape}") # [256] olmalı
    print(f"  label: {sample['labels']}") # 0-5 arası bir sayı olmalı

    print("\n--- Adım 3 tamamlandı. Şimdi Adım 4'e (DataLoader) geçebilirsiniz. ---")

except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Bu bloğu çalıştırmadan önce Adım 1+2 bloğunu başarıyla çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

Adım 3: BERT Tokenizer ('bert-base-uncased') yükleniyor...
Tokenizer başarıyla yüklendi.

NewsBertDataset sınıfı oluşturuluyor...
Dataset'ler MAX_LEN=256 ile oluşturuldu.

Eğitim setinden ilk örnek (BERT formatında):
  input_ids shape: torch.Size([256])
  attention_mask shape: torch.Size([256])
  label: 2

--- Adım 3 tamamlandı. Şimdi Adım 4'e (DataLoader) geçebilirsiniz. ---


In [5]:
# ==========================================================
# ADIM 4: DataLoader'ların Oluşturulması
# ==========================================================
# Bu adım, 'NewsBertDataset' sınıflarını
# modelin mini-gruplar halinde tüketebileceği bir formata sokar.
# Adım 3'te oluşturulan 'train_dataset_bert' ve 'test_dataset_bert' değişkenlerini kullanır.

try:
    # Mini-grup (batch) boyutu.
    # BERT modelleri büyük bellek kullandığı için 16 gibi daha küçük bir
    # batch size ile başlamak genellikle iyi bir fikirdir.
    BATCH_SIZE = 16

    # Eğitim verisi için DataLoader
    # dataset=train_dataset_bert: Adım 3'te oluşturduğumuz sınıfı kullanır.
    # shuffle=True: Her eğitim turunda (epoch) veriyi karıştırır.
    train_loader_bert = DataLoader(
        dataset=train_dataset_bert,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    # Test verisi için DataLoader
    # dataset=test_dataset_bert: Adım 3'te oluşturduğumuz sınıfı kullanır.
    # shuffle=False: Test ederken veriyi karıştırmaya gerek yoktur.
    test_loader_bert = DataLoader(
        dataset=test_dataset_bert,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    print(f"Adım 4: DataLoader'lar BATCH_SIZE={BATCH_SIZE} ile başarıyla oluşturuldu.")
    print(f"Eğitim DataLoader'ında yaklaşık {len(train_loader_bert)} mini-grup (batch) var.")
    print(f"Test DataLoader'ında yaklaşık {len(test_loader_bert)} mini-grup (batch) var.")

    # --- Test ---
    # Bir mini-grubu (batch) çekip boyutlarını kontrol edelim
    print("\nBir eğitim mini-grubu (batch) test ediliyor...")
    data_iter_bert = iter(train_loader_bert)
    batch = next(data_iter_bert)

    # Dataset'ten dönen sözlük formatını kontrol et
    print(f"  input_ids boyutu: {batch['input_ids'].shape}")
    print(f"  attention_mask boyutu: {batch['attention_mask'].shape}")
    print(f"  labels boyutu: {batch['labels'].shape}")

    print(f"(Beklenen input_ids boyutu: [{BATCH_SIZE}, {MAX_LEN}])")
    print(f"(Beklenen labels boyutu: [{BATCH_SIZE}])")

    print("\n--- Adım 4 tamamlandı. Şimdi Adım 5'e (Modeli Başlatma) geçebilirsiniz. ---")


except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Bu bloğu çalıştırmadan önce Adım 1+2 ve Adım 3 bloklarını başarıyla çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

Adım 4: DataLoader'lar BATCH_SIZE=16 ile başarıyla oluşturuldu.
Eğitim DataLoader'ında yaklaşık 6807 mini-grup (batch) var.
Test DataLoader'ında yaklaşık 1702 mini-grup (batch) var.

Bir eğitim mini-grubu (batch) test ediliyor...
  input_ids boyutu: torch.Size([16, 256])
  attention_mask boyutu: torch.Size([16, 256])
  labels boyutu: torch.Size([16])
(Beklenen input_ids boyutu: [16, 256])
(Beklenen labels boyutu: [16])

--- Adım 4 tamamlandı. Şimdi Adım 5'e (Modeli Başlatma) geçebilirsiniz. ---


In [6]:
# ==========================================================
# ADIM 5: Modelin ve Optimize Edicinin Başlatılması
# ==========================================================
# Bu adım, 'BertForSequenceClassification.from_pretrained' ve
# 'AdamW' satırlarına karşılık gelir.
# Adım 1+2'deki 'num_classes' (6 olan) değişkenini ve
# Adım 3'teki 'cpu' veya 'cuda' cihaz bilgisini kullanır.

try:
    # 1. Cihazı (Device) Ayarlama
    # Bu, 'cpu' veya 'cuda' olabilir.
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Adım 5: Model '{device}' cihazı üzerinde başlatılacak.")

    # 2. Modeli Yükleme
    # Slayt 41'deki BERT modelini yüklüyoruz.
    # 'bert-base-uncased' Slayt 41'de bahsedilen varyantlardan biridir.
    model_bert = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        # EN ÖNEMLİ DEĞİŞİKLİK: Sınıf sayısını 6 olarak ayarlıyoruz
        num_labels=num_classes  # 'num_classes' Adım 1+2'de 6 olarak hesaplanmıştı
    )

    # Modeli 'cpu' veya 'cuda' cihazına taşı
    model_bert = model_bert.to(device)
    print(f"Model ('bert-base-uncased') {num_classes} etiket (sınıf) için başarıyla yüklendi.")

    # 3. Optimize Ediciyi (Optimizer) Başlatma
    # AdamW, Transformer modelleri (Slayt 33) için standart optimize edicidir.
    # lr=2e-5: BERT ince ayarı (fine-tuning) için yaygın bir öğrenme oranıdır.
    optimizer_bert = AdamW(model_bert.parameters(), lr=2e-5)

    print("Optimize Edici (AdamW) başarıyla tanımlandı.")
    print("\n--- Adım 5 tamamlandı. Şimdi Adım 6'ya (Eğitim Fonksiyonları) geçebilirsiniz. ---")


except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Bu bloğu çalıştırmadan önce Adım 1+2 bloğunu başarıyla çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

Adım 5: Model 'cpu' cihazı üzerinde başlatılacak.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model ('bert-base-uncased') 6 etiket (sınıf) için başarıyla yüklendi.
Optimize Edici (AdamW) başarıyla tanımlandı.

--- Adım 5 tamamlandı. Şimdi Adım 6'ya (Eğitim Fonksiyonları) geçebilirsiniz. ---


In [7]:
# ==========================================================
# ADIM 6: Eğitim ve Değerlendirme Fonksiyonları
# ==========================================================
# Bu bölüm, 'train_epoch' ve 'evaluate' fonksiyonlarını tanımlar.
# Bu fonksiyonlar Adım 7'deki ana eğitim döngüsünde kullanılacaktır.
# Bu fonksiyonlar Adım 4'teki '...loader_bert' ve Adım 5'teki
# 'model_bert', 'optimizer_bert', 'device' değişkenlerini kullanır.

try:
    # --- Eğitim Fonksiyonu ---
    def train_epoch(model, loader, optimizer, device):
        model.train() # Modeli "eğitim modu"na al
        total_loss = 0
        correct = 0
        total = 0

        # Adım 4'te oluşturulan 'train_loader_bert'i kullan
        for batch in tqdm(loader, desc="Eğitim Adımı (Training)"):
            # Veri grubundaki (batch) tensörleri al
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Gradyanları sıfırla
            optimizer.zero_grad()

            # Modeli çalıştır (Forward pass)
            # 'labels' parametresini verdiğimizde model kaybı (loss) da hesaplar
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            # Kaybı (loss) al
            loss = outputs.loss

            # Geriye yayılım (Backpropagation)
            loss.backward()
            optimizer.step()

            # İstatistikleri kaydet
            total_loss += loss.item()
            predictions = torch.argmax(outputs.logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        # Epoch sonu ortalamaları
        return total_loss / len(loader), correct / total

    # --- Değerlendirme Fonksiyonu ---
    def evaluate(model, loader, device):
        model.eval() # Modeli "değerlendirme modu"na al (Dropout'u kapatır)
        total_loss = 0
        correct = 0
        total = 0

        # Gradyan hesaplamasını durdur
        with torch.no_grad():
            # Adım 4'te oluşturulan 'test_loader_bert'i kullan
            for batch in tqdm(loader, desc="Değerlendirme (Evaluating)"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                # Modeli çalıştır
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                # Kaybı ve istatistikleri kaydet
                loss = outputs.loss
                total_loss += loss.item()
                predictions = torch.argmax(outputs.logits, dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)

        # Epoch sonu ortalamaları
        return total_loss / len(loader), correct / total

    print("Adım 6: 'train_epoch' ve 'evaluate' fonksiyonları başarıyla tanımlandı.")
    print("\n--- Adım 6 tamamlandı. Şimdi Adım 7'ye (Ana Eğitim Döngüsü) geçebilirsiniz. ---")

except Exception as e:
    print(f"\nBir hata oluştu: {e}")

Adım 6: 'train_epoch' ve 'evaluate' fonksiyonları başarıyla tanımlandı.

--- Adım 6 tamamlandı. Şimdi Adım 7'ye (Ana Eğitim Döngüsü) geçebilirsiniz. ---


In [ ]:
# ==========================================================
# ADIM 7: Ana Eğitim Döngüsü
# ==========================================================
# Bu adım, '===== TRAIN MODEL =====' bölümüne karşılık gelir.
# Adım 6'da tanımlanan 'train_epoch' ve 'evaluate'
# fonksiyonlarını çağırır.

try:
    # Epoch sayısı. BERT ince ayarı (fine-tuning) için 2-4 arası yeterlidir.
    # 'cpu' üzerinde olduğunuz için 1 veya 2 ile başlamak isteyebilirsiniz.
    EPOCHS = 2

    print(f"Adım 7: Ana eğitim döngüsü başlıyor... Toplam {EPOCHS} epoch sürecek.")
    print(f"Kullanılan cihaz: {device}. Bu işlem UZUN sürebilir...")

    for epoch in range(EPOCHS):
        print(f"\n===== Epoch {epoch + 1}/{EPOCHS} =====")

        # Modeli eğit (Adım 6'daki fonksiyon)
        train_loss, train_acc = train_epoch(
            model_bert,
            train_loader_bert,
            optimizer_bert,
            device
        )

        print(f"Epoch {epoch + 1} Eğitim Sonucu:")
        print(f"  Ortalama Eğitim Kaybı (Loss): {train_loss:.4f}")
        print(f"  Eğitim Doğruluğu (Accuracy): {train_acc * 100:.2f}%")

        # Modeli değerlendir (Adım 6'daki fonksiyon)
        test_loss, test_acc = evaluate(
            model_bert,
            test_loader_bert,
            device
        )

        print(f"Epoch {epoch + 1} Değerlendirme Sonucu:")
        print(f"  Ortalama Test Kaybı (Loss): {test_loss:.4f}")
        print(f"  Test Doğruluğu (Accuracy): {test_acc * 100:.2f}%")

    print(f"\n--- Adım 7 (Eğitim) {EPOCHS} epoch için tamamlandı. ---")
    print("Şimdi Adım 8'e (Tahmin Fonksiyonu) geçebilirsiniz.")


except NameError as e:
    print("\n--- HATA ---")
    print(f"'{e.name}' değişkeni tanımlanmamış görünüyor.")
    print("Bu bloğu çalıştırmadan önce Adım 4, 5 ve 6 bloklarını başarıyla çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"\nBir hata oluştu: {e}")

Adım 7: Ana eğitim döngüsü başlıyor... Toplam 2 epoch sürecek.
Kullanılan cihaz: cpu. Bu işlem UZUN sürebilir...

===== Epoch 1/2 =====


Eğitim Adımı (Training):   0%|          | 14/6807 [12:19<99:36:45, 52.79s/it]


KeyboardInterrupt: 